# Phase Final: RemoteCLIP Cross-Attention Training

This notebook trains the final RemoteCLIP-based change captioning model on the LEVIR-CC dataset. It uses the shared dataset loader format, then adds cross-attention fusion and contrastive caption alignment.

## 1. Project & Environment Setup

This notebook trains the phase-final RemoteCLIP model on the LEVIR-CC dataset. It uses the shared dataset loader format, then adds cross-attention fusion and contrastive caption alignment.

In [1]:
import json
import os
from pathlib import Path

import torch

from src.config import DataConfig, ModelConfig, RemoteCLIPConfig, TrainConfig
from src.dataset import (
    build_remoteclip_transforms,
    build_vocabulary_from_multiple_annotations,
    get_levircc_loaders,
    get_secondcc_loaders,
    load_levircc_annotations,
    split_samples_by_split,
)
from src.metrics import SentenceEmbeddingScorer, evaluate_model_on_loader
from src.models.final_model import (
    RemoteCLIPCrossAttentionModel,
    phase5_total_loss,
)
from src.training import (
    build_criterion,
    build_optimizer_and_scheduler,
    load_checkpoint,
    save_checkpoint,
    validate,
    visualize_predictions,
)
from src.utils import get_device, set_seed, setup_project_path

PROJECT_ROOT = setup_project_path()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

set_seed(42)
device = get_device()
print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {device}')

/home2/sankalp0109/src_IS/venv/lib/python3.10/site-packages/torch/cuda/__init__.py:262: UserWarning: 
    Found GPU0 NVIDIA GeForce GTX 1080 Ti which is of cuda capability 6.1.
    PyTorch no longer supports this GPU because it is too old.
    The minimum cuda capability supported by this library is 7.5.
    
  warnings.warn(
/home2/sankalp0109/src_IS/venv/lib/python3.10/site-packages/torch/cuda/__init__.py:287: UserWarning: 
NVIDIA GeForce GTX 1080 Ti with CUDA capability sm_61 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_75 sm_80 sm_86 sm_90 sm_100 sm_120 compute_120.
If you want to use the NVIDIA GeForce GTX 1080 Ti GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


Project root: /home2/sankalp0109/src_IS
Device: cpu


/home2/sankalp0109/src_IS/src/utils.py:54: RuntimeWarning: CUDA initialized but cannot execute kernels on this GPU; falling back to CPU. Original error: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

  warnings.warn(


## 2. Prepare Config Objects and Paths

In [2]:
data_cfg = DataConfig(batch_size=8, val_batch_size=16)
model_cfg = ModelConfig()
remoteclip_cfg = RemoteCLIPConfig()
train_cfg = TrainConfig()
RESEARCHER_NAME = 'phase_final_remoteclip_difference'

remoteclip_cfg.download_if_missing = os.environ.get(
    'REMOTECLIP_DOWNLOAD_IF_MISSING', '0'
).strip().lower() in {'1', 'true', 'yes'}

CHECKPOINT_DIR = data_cfg.checkpoint_dir
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
current_path = CHECKPOINT_DIR / f'{RESEARCHER_NAME}_current.pt'
best_path = CHECKPOINT_DIR / f'{RESEARCHER_NAME}_best.pt'
metrics_path = CHECKPOINT_DIR / f'{RESEARCHER_NAME}_metrics.json'
metadata_path = CHECKPOINT_DIR / f'{RESEARCHER_NAME}_metadata.json'
vocab_artifact_path = CHECKPOINT_DIR / f'{RESEARCHER_NAME}_vocab.json'

print(data_cfg)
print(model_cfg)
print(remoteclip_cfg)
print(train_cfg)

DataConfig(data_root=PosixPath('Levir-CC-dataset'), caption_json=PosixPath('Levir-CC-dataset/LevirCCcaptions.json'), image_root=PosixPath('Levir-CC-dataset/images'), checkpoint_dir=PosixPath('checkpoints'), vocab_path=PosixPath('checkpoints/shared_vocab.pkl'), img_size=(256, 256), batch_size=8, val_batch_size=16, num_workers=0, min_word_freq=2, caption_index=0, imagenet_mean=(0.485, 0.456, 0.406), imagenet_std=(0.229, 0.224, 0.225))
ModelConfig(encoder_dim=512, embed_dim=256, num_heads=4, num_decoder_layers=2, max_caption_len=100, dropout=0.1, encoder_hidden_dim=64)
RemoteCLIPConfig(model_name='ViT-B-32', checkpoint_path=PosixPath('checkpoints/RemoteCLIP-ViT-B-32.pt'), hf_repo_id='chendelong/RemoteCLIP', encoder_dim=512, freeze_backbone=True, download_if_missing=True, fusion_dropout=0.1, image_size=(224, 224), image_mean=(0.48145466, 0.4578275, 0.40821073), image_std=(0.26862954, 0.26130258, 0.27577711))
TrainConfig(learning_rate=0.0001, weight_decay=1e-05, num_epochs=15, grad_clip=1.0

## 3. Build RemoteCLIP Preprocessing Transforms

In [3]:
remoteclip_transform = build_remoteclip_transforms(
    img_size=remoteclip_cfg.image_size,
    mean=remoteclip_cfg.image_mean,
    std=remoteclip_cfg.image_std,
)
print(remoteclip_transform)

Compose(
    Resize(size=(224, 224), interpolation=bicubic, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)


## 4. Inspect Dataset Format

`get_levircc_loaders` expects a caption JSON file with an `images` list. Each sample contains the image filename, a relative path segment, a split label, a change flag, and one or more caption sentences under `sentences`. The loader uses `caption_index` to select which caption to train on when multiple captions are available.

In [4]:
annotations = load_levircc_annotations(data_cfg.caption_json)
train_samples, val_samples, test_samples = split_samples_by_split(annotations)

print(f'Caption JSON: {data_cfg.caption_json}')
print(f'Image root: {data_cfg.image_root}')
print(f'Total image records: {len(annotations["images"])}')
print(f'Train / Val / Test: {len(train_samples)} / {len(val_samples)} / {len(test_samples)}')

for index, sample in enumerate(annotations['images'][:3], start=1):
    print(f'\nSample {index}')
    print(json.dumps({
        'filepath': sample.get('filepath'),
        'filename': sample.get('filename'),
        'split': sample.get('split'),
        'changeflag': sample.get('changeflag'),
        'num_sentences': len(sample.get('sentences', [])),
        'first_caption': sample.get('sentences', [{}])[0].get('raw', ''),
    }, indent=2))

Caption JSON: Levir-CC-dataset/LevirCCcaptions.json
Image root: Levir-CC-dataset/images
Total image records: 10077
Train / Val / Test: 6815 / 1333 / 1929

Sample 1
{
  "filepath": "train",
  "filename": "train_000001.png",
  "split": "train",
  "changeflag": 0,
  "num_sentences": 5,
  "first_caption": " there is no difference ."
}

Sample 2
{
  "filepath": "train",
  "filename": "train_000002.png",
  "split": "train",
  "changeflag": 0,
  "num_sentences": 5,
  "first_caption": " there is no difference ."
}

Sample 3
{
  "filepath": "train",
  "filename": "train_000003.png",
  "split": "train",
  "changeflag": 0,
  "num_sentences": 5,
  "first_caption": " there is no difference ."
}


In [5]:
SECONDCC_ROOT = PROJECT_ROOT / "SECOND-CC-AUG"

secondcc_caption_json = SECONDCC_ROOT / "SECOND-CC-AUG.json"
secondcc_image_root = SECONDCC_ROOT 

print(f"SECOND-CC caption JSON: {secondcc_caption_json}")
print(f"SECOND-CC image root: {secondcc_image_root}")
second_annotations = load_levircc_annotations(secondcc_caption_json)
shared_vocab = build_vocabulary_from_multiple_annotations(
    [annotations, second_annotations],
    min_freq=data_cfg.min_word_freq,
)
print(f'SECOND-CC caption JSON: {secondcc_caption_json}')
print(f'SECOND-CC image root: {secondcc_image_root}')
print(f'Shared vocabulary size: {len(shared_vocab.word2idx)}')

SECOND-CC caption JSON: /home2/sankalp0109/src_IS/SECOND-CC-AUG/SECOND-CC-AUG.json
SECOND-CC image root: /home2/sankalp0109/src_IS/SECOND-CC-AUG


SECOND-CC caption JSON: /home2/sankalp0109/src_IS/SECOND-CC-AUG/SECOND-CC-AUG.json
SECOND-CC image root: /home2/sankalp0109/src_IS/SECOND-CC-AUG
Shared vocabulary size: 1300


## 5. Create DataLoaders with `get_levircc_loaders`

In [6]:
train_loader, val_loader, test_loader, vocab = get_levircc_loaders(
    caption_json=data_cfg.caption_json,
    image_root=data_cfg.image_root,
    vocab=shared_vocab,
    batch_size=data_cfg.batch_size,
    val_batch_size=data_cfg.val_batch_size,
    device=str(device),
    min_word_freq=data_cfg.min_word_freq,
    caption_index=data_cfg.caption_index,
    num_workers=data_cfg.num_workers,
    vocab_path=None,
    transforms_fn=remoteclip_transform,
)

print(f'Vocabulary size: {len(vocab.word2idx)}')
print(f'Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

Vocabulary size: 1300
Train: 852 | Val: 84 | Test: 121


## 6. Build Phase 5 Model (RemoteCLIP Cross-Attention)

In [7]:
model = RemoteCLIPCrossAttentionModel(
    vocab_size=len(vocab.word2idx),
    encoder_dim=remoteclip_cfg.encoder_dim,
    embed_dim=model_cfg.embed_dim,
    num_heads=model_cfg.num_heads,
    num_decoder_layers=model_cfg.num_decoder_layers,
    max_caption_len=model_cfg.max_caption_len,
    dropout=model_cfg.dropout,
    pad_idx=vocab.pad_idx,
    remoteclip_model_name=remoteclip_cfg.model_name,
    remoteclip_checkpoint_path=remoteclip_cfg.checkpoint_path,
    freeze_remoteclip=remoteclip_cfg.freeze_backbone,
    download_if_missing=remoteclip_cfg.download_if_missing,
    remoteclip_repo_id=remoteclip_cfg.hf_repo_id,
    fusion_heads=model_cfg.num_heads,
    token_count=4,
    contrastive_dim=model_cfg.embed_dim,
).to(device)

trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
total = sum(parameter.numel() for parameter in model.parameters())
print(f'Model: {model.model_name}')
print(f'Trainable parameters: {trainable:,} / {total:,}')

Model: RemoteCLIPCrossAttentionModel
Trainable parameters: 9,111,572 / 160,388,885


## 7. Load or Resume From Checkpoint

In [8]:
criterion = build_criterion(vocab)
optimizer, scheduler = build_optimizer_and_scheduler(model, train_cfg)
best_val_loss = float('inf')
start_epoch = 1

if current_path.exists():
    model, optimizer, last_epoch, loaded_vocab, loaded_loss = load_checkpoint(
        model, optimizer, current_path, device
    )
    start_epoch = last_epoch + 1
    if loaded_loss is not None:
        best_val_loss = loaded_loss

print(f'Start epoch: {start_epoch}')
print(f'Best validation loss: {best_val_loss}')

Checkpoint loaded from checkpoints/phase_final_remoteclip_difference_current.pt


  Epoch: 15, Loss: 1.0357


Start epoch: 16
Best validation loss: 1.0357029780837668


## 8. Training Loop

In [9]:
optimizer, scheduler = build_optimizer_and_scheduler(model, train_cfg)
training_history = []
best_checkpoint_info = {
    'epoch': 0,
    'val_loss': float('inf'),
    'path': None,
}

for epoch in range(start_epoch, train_cfg.num_epochs + 1):
    model.train()
    total_loss = 0.0
    total_caption_loss = 0.0
    total_contrastive_loss = 0.0
    num_batches = 0

    for batch_idx, batch in enumerate(train_loader):
        images = batch['images'].to(device, non_blocking=True)
        caption_tokens = batch['caption_tokens'].to(device, non_blocking=True)
        input_tokens = caption_tokens[:, :-1]
        target_tokens = caption_tokens[:, 1:]

        outputs = model(images, input_tokens, return_aux=True)
        total_batch_loss, caption_batch_loss, contrastive_batch_loss = phase5_total_loss(
            logits=outputs['logits'],
            target_tokens=target_tokens,
            criterion=criterion,
            image_embeddings=outputs['image_embeddings'],
            text_embeddings=outputs['text_embeddings'],
            contrastive_weight=0.1,
            temperature=0.07,
            pad_idx=vocab.pad_idx,
        )

        optimizer.zero_grad()
        total_batch_loss.backward()
        if train_cfg.grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
        optimizer.step()

        total_loss += total_batch_loss.item()
        total_caption_loss += caption_batch_loss.item()
        total_contrastive_loss += contrastive_batch_loss.item()
        num_batches += 1

        if train_cfg.log_every and (batch_idx + 1) % train_cfg.log_every == 0:
            print(
                f'  Batch {batch_idx + 1}/{len(train_loader)}: '
                f'loss={total_loss / num_batches:.4f}, '
                f'caption={total_caption_loss / num_batches:.4f}, '
                f'contrastive={total_contrastive_loss / num_batches:.4f}',
                flush=True,
            )

    train_loss = total_loss / max(num_batches, 1)
    val_loss, val_accuracy = validate(
        model,
        val_loader,
        criterion,
        device,
        pad_idx=vocab.pad_idx,
        return_accuracy=True,
    )
    scheduler.step()

    print(
        f'Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}, token_acc={val_accuracy:.4f}',
        flush=True,
    )

    save_checkpoint(
        model,
        optimizer,
        epoch,
        val_loss,
        vocab,
        CHECKPOINT_DIR,
        filename=current_path.name,
        extra={'phase': 5, 'model': 'RemoteCLIPCrossAttentionModel', 'fusion': 'cross_attention', 'loss': 'caption_plus_contrastive'},
    )

    if val_loss < best_checkpoint_info['val_loss']:
        best_checkpoint_info = {
            'epoch': epoch,
            'val_loss': val_loss,
            'path': str(best_path),
        }
        save_checkpoint(
            model,
            optimizer,
            epoch,
            val_loss,
            vocab,
            CHECKPOINT_DIR,
            filename=best_path.name,
            extra={'phase': 5, 'model': 'RemoteCLIPCrossAttentionModel', 'fusion': 'cross_attention', 'loss': 'caption_plus_contrastive'},
        )

    training_history.append(
        {
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_accuracy': val_accuracy,
            'caption_loss': total_caption_loss / max(num_batches, 1),
            'contrastive_loss': total_contrastive_loss / max(num_batches, 1),
        }
    )

print('Training complete.')
print(best_checkpoint_info)

Training complete.
{'epoch': 0, 'val_loss': inf, 'path': None}


## 9. Evaluation on Test Set and Semantic Scoring

In [10]:
# Load the best model checkpoint for evaluation
if best_path.exists():
    model, _, _, _, _ = load_checkpoint(model, None, best_path, device)
    print(f'Loaded best model from {best_path} for Stage 1 evaluation.')

levir_stage1_test_loss = validate(model, test_loader, criterion, device, pad_idx=vocab.pad_idx)
print(f'LEVIR-CC test loss: {levir_stage1_test_loss:.4f}')

levir_semantic_scorer = SentenceEmbeddingScorer('all-MiniLM-L6-v2', device=str(device))
levir_stage1_metrics, levir_stage1_details = evaluate_model_on_loader(
    model,
    test_loader,
    vocab,
    device,
    semantic_model=levir_semantic_scorer,
)

for name, value in levir_stage1_metrics.items():
    if isinstance(value, float):
        print(f'{name}: {value:.4f}')
    else:
        print(f'{name}: {value}')

print(f'LEVIR-CC semantic details: {len(levir_stage1_details)} samples')

LEVIR-CC test loss: 0.8982


BLEU-1: 0.5311
BLEU-2: 0.4355
BLEU-3: 0.3810
BLEU-4: 0.3365
METEOR: 0.6392
ROUGE-L: 0.6604
CIDEr: 5.3999
Semantic-Similarity: 0.7123
num_samples: 1929
LEVIR-CC semantic details: 1929 samples


## 10. SECOND-CC Dataset Setup and Sequential Fine-Tuning

In [11]:
SECONDCC_ROOT = PROJECT_ROOT / "SECOND-CC-AUG"

secondcc_caption_json = SECONDCC_ROOT / "SECOND-CC-AUG.json"
secondcc_image_root = SECONDCC_ROOT 

print(f"SECOND-CC caption JSON: {secondcc_caption_json}")
print(f"SECOND-CC image root: {secondcc_image_root}")

SECOND-CC caption JSON: /home2/sankalp0109/src_IS/SECOND-CC-AUG/SECOND-CC-AUG.json
SECOND-CC image root: /home2/sankalp0109/src_IS/SECOND-CC-AUG


In [12]:
secondcc_transform = build_remoteclip_transforms(
    img_size=remoteclip_cfg.image_size,
    mean=remoteclip_cfg.image_mean,
    std=remoteclip_cfg.image_std,
)

second_train_loader, second_val_loader, second_test_loader, second_vocab = get_secondcc_loaders(
    caption_json=secondcc_caption_json,
    image_root=secondcc_image_root,
    vocab=vocab,
    batch_size=data_cfg.batch_size,
    val_batch_size=data_cfg.val_batch_size,
    device=str(device),
    min_word_freq=data_cfg.min_word_freq,
    caption_index=data_cfg.caption_index,
    num_workers=data_cfg.num_workers,
    vocab_path=data_cfg.vocab_path,
    transforms_fn=secondcc_transform,
)

print(f'SECOND-CC vocabulary size: {len(second_vocab.word2idx)}')
print(
    f'SECOND-CC Train: {len(second_train_loader)} | Val: {len(second_val_loader)} | Test: {len(second_test_loader)}'
)

SECOND-CC vocabulary size: 1300
SECOND-CC Train: 1055 | Val: 75 | Test: 77


## 11. SECOND-CC Fine-Tuning Loop

In [13]:
second_criterion = build_criterion(vocab)
second_optimizer, second_scheduler = build_optimizer_and_scheduler(model, train_cfg)

second_training_history = []

second_best_checkpoint_info = {
    "epoch": 0,
    "val_loss": float("inf"),
    "path": str(CHECKPOINT_DIR / f"{RESEARCHER_NAME}_secondcc_best.pt"),
}

start_epoch = 1

second_current_path = CHECKPOINT_DIR / f"{RESEARCHER_NAME}_secondcc_current.pt"

if second_current_path.exists():
    model, second_optimizer, last_epoch, loaded_vocab, loaded_loss = load_checkpoint(
        model,
        second_optimizer,
        second_current_path,
        device,
    )
    start_epoch = last_epoch + 1
    if loaded_loss is not None:
        second_best_checkpoint_info["val_loss"] = loaded_loss
    second_best_checkpoint_info["epoch"] = last_epoch
    second_best_checkpoint_info["path"] = str(
        CHECKPOINT_DIR / f"{RESEARCHER_NAME}_secondcc_best.pt"
    )
else:
    if best_path.exists():
        model, _, _, _, _ = load_checkpoint(model, None, best_path, device)
        print(f"Loaded best Levir-CC model from {best_path} to start SECOND-CC fine-tuning.\")

print(f"SECOND-CC start epoch: {start_epoch}")
print(f"Best validation loss: {second_best_checkpoint_info['val_loss']}")

for epoch in range(start_epoch, train_cfg.num_epochs):
    model.train()
    total_loss = 0.0
    total_caption_loss = 0.0
    total_contrastive_loss = 0.0
    num_batches = 0

    for batch_idx, batch in enumerate(second_train_loader):
        images = batch['images'].to(device, non_blocking=True)
        caption_tokens = batch['caption_tokens'].to(device, non_blocking=True)
        input_tokens = caption_tokens[:, :-1]
        target_tokens = caption_tokens[:, 1:]

        outputs = model(images, input_tokens, return_aux=True)
        total_batch_loss, caption_batch_loss, contrastive_batch_loss = phase5_total_loss(
            logits=outputs['logits'],
            target_tokens=target_tokens,
            criterion=second_criterion,
            image_embeddings=outputs['image_embeddings'],
            text_embeddings=outputs['text_embeddings'],
            contrastive_weight=0.1,
            temperature=0.07,
            pad_idx=vocab.pad_idx,
        )

        second_optimizer.zero_grad()
        total_batch_loss.backward()
        if train_cfg.grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
        second_optimizer.step()

        total_loss += total_batch_loss.item()
        total_caption_loss += caption_batch_loss.item()
        total_contrastive_loss += contrastive_batch_loss.item()
        num_batches += 1

        if train_cfg.log_every and (batch_idx + 1) % train_cfg.log_every == 0:
            print(
                f'  Batch {batch_idx + 1}/{len(second_train_loader)}: '
                f'loss={total_loss / num_batches:.4f}, '
                f'caption={total_caption_loss / num_batches:.4f}, '
                f'contrastive={total_contrastive_loss / num_batches:.4f}',
                flush=True,
            )

    train_loss = total_loss / max(num_batches, 1)
    val_loss, val_accuracy = validate(
        model,
        second_val_loader,
        second_criterion,
        device,
        pad_idx=vocab.pad_idx,
        return_accuracy=True,
    )
    second_scheduler.step()

    print(
        f'SECOND-CC Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}, token_acc={val_accuracy:.4f}',
        flush=True,
    )

    save_checkpoint(
        model,
        second_optimizer,
        epoch,
        val_loss,
        vocab,
        CHECKPOINT_DIR,
        filename=f'{RESEARCHER_NAME}_secondcc_current.pt',
        extra={'phase': 5, 'dataset': 'SECOND-CC', 'model': 'RemoteCLIPCrossAttentionModel', 'fusion': 'cross_attention', 'loss': 'caption_plus_contrastive'},
    )

    if val_loss < second_best_checkpoint_info['val_loss']:
        second_best_checkpoint_info = {
            'epoch': epoch,
            'val_loss': val_loss,
            'path': str(CHECKPOINT_DIR / f'{RESEARCHER_NAME}_secondcc_best.pt'),
        }
        save_checkpoint(
            model,
            second_optimizer,
            epoch,
            val_loss,
            vocab,
            CHECKPOINT_DIR,
            filename=f'{RESEARCHER_NAME}_secondcc_best.pt',
            extra={'phase': 5, 'dataset': 'SECOND-CC', 'model': 'RemoteCLIPCrossAttentionModel', 'fusion': 'cross_attention', 'loss': 'caption_plus_contrastive'},
        )

    second_training_history.append(
        {
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_accuracy': val_accuracy,
            'caption_loss': total_caption_loss / max(num_batches, 1),
            'contrastive_loss': total_contrastive_loss / max(num_batches, 1),
        }
    )

print('SECOND-CC fine-tuning complete.')
print(second_best_checkpoint_info)

SECOND-CC start epoch: 1
Best validation loss: inf


  SECOND-CC Batch 10/1055: loss=4.8224, caption=4.5463, contrastive=2.7612


  SECOND-CC Batch 20/1055: loss=4.4633, caption=4.2107, contrastive=2.5262


  SECOND-CC Batch 30/1055: loss=4.2235, caption=3.9826, contrastive=2.4093


  SECOND-CC Batch 40/1055: loss=4.0031, caption=3.7660, contrastive=2.3703


...

  SECOND-CC Batch 1040/1055: loss=1.2747, caption=1.2220, contrastive=0.5269


  SECOND-CC Batch 1050/1055: loss=1.2747, caption=1.2220, contrastive=0.5274


SECOND-CC Epoch 14: train=1.2748, val=1.5509, token_acc=0.6131


Checkpoint saved: checkpoints/phase_final_remoteclip_difference_secondcc_current.pt


SECOND-CC fine-tuning complete.
{'epoch': 6, 'val_loss': 1.5224345795957681, 'path': 'checkpoints/phase_final_remoteclip_difference_secondcc_best.pt'}


## 12. Post-Finetune Evaluation on Both Test Sets

In [14]:
# Load the best SECOND-CC fine-tuned model checkpoint for evaluation
second_best_path = CHECKPOINT_DIR / f'{RESEARCHER_NAME}_secondcc_best.pt'
if second_best_path.exists():
    model, _, _, _, _ = load_checkpoint(model, None, second_best_path, device)
    print(f'Loaded best SECOND-CC model from {second_best_path} for final evaluation.')

levir_post_finetune_test_loss = validate(model, test_loader, second_criterion, device, pad_idx=vocab.pad_idx)
second_test_loss = validate(model, second_test_loader, second_criterion, device, pad_idx=vocab.pad_idx)

print(f'LEVIR-CC test loss after SECOND-CC fine-tuning: {levir_post_finetune_test_loss:.4f}')
print(f'SECOND-CC test loss: {second_test_loss:.4f}')

shared_semantic_scorer = SentenceEmbeddingScorer('all-MiniLM-L6-v2', device=str(device))
levir_post_metrics, levir_post_details = evaluate_model_on_loader(
    model,
    test_loader,
    vocab,
    device,
    semantic_model=shared_semantic_scorer,
)
second_test_metrics, second_test_details = evaluate_model_on_loader(
    model,
    second_test_loader,
    vocab,
    device,
    semantic_model=shared_semantic_scorer,
)

print('\nLEVIR-CC metrics after SECOND-CC fine-tuning')
for name, value in levir_post_metrics.items():
    if isinstance(value, float):
        print(f'{name}: {value:.4f}')
    else:
        print(f'{name}: {value}')

print('\nSECOND-CC metrics')
for name, value in second_test_metrics.items():
    if isinstance(value, float):
        print(f'{name}: {value:.4f}')
    else:
        print(f'{name}: {value}')

print(f'LEVIR-CC semantic details: {len(levir_post_details)} samples')
print(f'SECOND-CC semantic details: {len(second_test_details)} samples')

LEVIR-CC test loss after SECOND-CC fine-tuning: 4.0801
SECOND-CC test loss: 1.5856



LEVIR-CC metrics after SECOND-CC fine-tuning
BLEU-1: 0.1677
BLEU-2: 0.1610
BLEU-3: 0.1561
BLEU-4: 0.1489
METEOR: 0.3935
ROUGE-L: 0.4369
CIDEr: 4.0598
Semantic-Similarity: 0.4581
num_samples: 1929

SECOND-CC metrics
BLEU-1: 0.3627
BLEU-2: 0.2452
BLEU-3: 0.1789
BLEU-4: 0.1295
METEOR: 0.4131
ROUGE-L: 0.4334
CIDEr: 2.9234
Semantic-Similarity: 0.5498
num_samples: 1227
LEVIR-CC semantic details: 1929 samples
SECOND-CC semantic details: 1227 samples


## 10. Visualize and Export Predictions

In [15]:
visualize_predictions(
    model,
    test_loader,
    vocab,
    device,
    num_samples=300,
    output_pdf='predictions_phase_final.pdf',
    mean=remoteclip_cfg.image_mean,
    std=remoteclip_cfg.image_std,
)

# Optional quick inline inspection of the first batch predictions.
model.eval()
batch = next(iter(test_loader))
with torch.no_grad():
    images = batch['images'].to(device)
    caption_tokens = batch['caption_tokens'].to(device)
    logits = model(images, caption_tokens[:, :-1])
    sample_tokens = logits[0].argmax(dim=-1).tolist()
    print('Reference:', batch['captions'][0])
    print('Predicted token ids:', sample_tokens)

Saved 300 predictions to 'predictions_phase_final.pdf'.


Reference: there is no difference .
Predicted token ids: [4, 5, 6, 7, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


## 11. Save Final Artifacts

In [16]:
final_checkpoint = CHECKPOINT_DIR / f'{RESEARCHER_NAME}_final.pt'
final_metrics = {
    'levir_stage1': {
        'test_loss': float(levir_stage1_test_loss),
        'metrics': {key: float(value) if isinstance(value, (int, float)) else value for key, value in levir_stage1_metrics.items()},
    },
    'levir_post_secondcc': {
        'test_loss': float(levir_post_finetune_test_loss),
        'metrics': {key: float(value) if isinstance(value, (int, float)) else value for key, value in levir_post_metrics.items()},
    },
    'secondcc': {
        'test_loss': float(second_test_loss),
        'metrics': {key: float(value) if isinstance(value, (int, float)) else value for key, value in second_test_metrics.items()},
    },
    'training_history': training_history,
    'second_training_history': second_training_history,
    'best_checkpoint': best_checkpoint_info,
    'second_best_checkpoint': second_best_checkpoint_info,
}

save_checkpoint(
    model,
    second_optimizer,
    epoch=train_cfg.num_epochs,
    loss=second_test_loss,
    vocab=vocab,
    checkpoint_dir=CHECKPOINT_DIR,
    filename=final_checkpoint.name,
    extra={'phase': 5, 'model': 'RemoteCLIPCrossAttentionModel', 'stages': ['LEVIR-CC', 'SECOND-CC']},
)

with open(vocab_artifact_path, 'w', encoding='utf-8') as handle:
    json.dump(vocab.word2idx, handle, indent=2)

with open(metrics_path, 'w', encoding='utf-8') as handle:
    json.dump(final_metrics, handle, indent=2)

metadata = {
    'researcher_name': RESEARCHER_NAME,
    'phase': 5,
    'model_class': model.__class__.__name__,
    'remoteclip_model_name': remoteclip_cfg.model_name,
    'remoteclip_checkpoint_path': str(remoteclip_cfg.checkpoint_path),
    'download_if_missing': remoteclip_cfg.download_if_missing,
    'batch_size': data_cfg.batch_size,
    'val_batch_size': data_cfg.val_batch_size,
    'learning_rate': train_cfg.learning_rate,
    'weight_decay': train_cfg.weight_decay,
    'num_epochs': train_cfg.num_epochs,
    'grad_clip': train_cfg.grad_clip,
    'contrastive_weight': 0.1,
    'temperature': 0.07,
    'stages': ['LEVIR-CC', 'SECOND-CC'],
    'levir_caption_json': str(data_cfg.caption_json),
    'secondcc_caption_json': str(secondcc_caption_json),
    'levir_image_root': str(data_cfg.image_root),
    'secondcc_image_root': str(secondcc_image_root),
    'final_checkpoint': str(final_checkpoint),
    'best_checkpoint': best_checkpoint_info,
    'second_best_checkpoint': second_best_checkpoint_info,
}

with open(metadata_path, 'w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2)

print(f'Final checkpoint: {final_checkpoint}')
print(f'Vocab artifact: {vocab_artifact_path}')
print(f'Metrics JSON: {metrics_path}')
print(f'Metadata JSON: {metadata_path}')

Checkpoint saved: checkpoints/phase_final_remoteclip_difference_final.pt


Final checkpoint: checkpoints/phase_final_remoteclip_difference_final.pt
Vocab artifact: checkpoints/phase_final_remoteclip_difference_vocab.json
Metrics JSON: checkpoints/phase_final_remoteclip_difference_metrics.json
Metadata JSON: checkpoints/phase_final_remoteclip_difference_metadata.json
